# Skin Lesion Classifier — EfficientNet-B3 Fine-tuning

**Dataset:** DermaMNIST 7-class (64×64 pre-processed, upsampled to 224×224)  
**Model:** EfficientNet-B3 with 2-phase fine-tuning  
**Target:** Test macro-F1 ≥ 0.65 (baseline = 0.449)  

### How to use this notebook
1. Upload the `data/` folder from this package as a **Kaggle Dataset** named `dermamnist-64px`
2. Add it to this notebook via *Add Data → Your Datasets*
3. Enable **GPU** (T4 × 2 or P100) in *Settings → Accelerator*
4. Run all cells
5. Download `skin_cnn_torch.pt`, `image_labels.json`, `image_training_metrics.json` from the output
6. Copy all 3 files to `backend/models/` on your local machine
7. Restart Django — the model loads automatically

In [ ]:
import os, json, time, sys
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import models, transforms as T
from sklearn.metrics import f1_score, accuracy_score, classification_report
from pathlib import Path

# ── Reproducibility ───────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────────────
IMAGE_SIZE    = 224    # upsample 64→224 for EfficientNet
BATCH_SIZE    = 64     # reduce to 32 if OOM on T4
PHASE1_EPOCHS = 5      # head-only warmup
PHASE2_EPOCHS = 30     # fine-tune last 3 blocks
LR_HEAD       = 1e-3
LR_FINETUNE   = 3e-5
PATIENCE      = 8      # early stopping
LABEL_SMOOTH  = 0.1
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────
# Kaggle dataset path — adjust the dataset name if you named it differently
DATA_DIR = Path('/kaggle/input/dermamnist-64px')
if not DATA_DIR.exists():
    DATA_DIR = Path('data')  # local fallback
    print('WARNING: Using local data/ folder')

X_train = np.load(DATA_DIR / 'train_images.npy')  # (7007, 64, 64, 3) float32 [0,1]
y_train = np.load(DATA_DIR / 'train_labels.npy').astype(np.int64)
X_val   = np.load(DATA_DIR / 'val_images.npy')
y_val   = np.load(DATA_DIR / 'val_labels.npy').astype(np.int64)
X_test  = np.load(DATA_DIR / 'test_images.npy')
y_test  = np.load(DATA_DIR / 'test_labels.npy').astype(np.int64)

with open(DATA_DIR / 'label_map.json') as f:
    classes = json.load(f)
num_classes = len(classes)
counts = np.bincount(y_train)

print(f'Train : {X_train.shape}  |  Val : {X_val.shape}  |  Test : {X_test.shape}')
print(f'Classes ({num_classes}):')
for i, (c, n) in enumerate(zip(classes, counts)):
    bar = '█' * (n // 100)
    print(f'  [{i}] {c:45s} {n:5d}  {bar}')
print(f'Imbalance ratio: {counts.max()/counts.min():.1f}x')

In [ ]:
# ── Dataset class ─────────────────────────────────────────────────────────
class SkinDataset(Dataset):
    """Wraps pre-loaded NPY arrays with optional torchvision transforms."""
    def __init__(self, X: np.ndarray, y: np.ndarray, transform=None):
        # X: (N, H, W, C) float32 [0,1]
        self.X = torch.from_numpy(X).permute(0, 3, 1, 2)  # → (N, C, H, W)
        self.y = torch.from_numpy(y)
        self.transform = transform

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        img = self.X[idx]  # (C, H, W) float32
        if self.transform:
            img = self.transform(img)
        return img, self.y[idx]


# Augmentation pipeline for training
train_tfm = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE), antialias=True),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.5),
    T.RandomRotation(degrees=20),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    T.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    T.Normalize(mean=MEAN, std=STD),
])
eval_tfm = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE), antialias=True),
    T.Normalize(mean=MEAN, std=STD),
])

train_ds = SkinDataset(X_train, y_train, transform=train_tfm)
val_ds   = SkinDataset(X_val,   y_val,   transform=eval_tfm)
test_ds  = SkinDataset(X_test,  y_test,  transform=eval_tfm)

# WeightedRandomSampler — each class gets equal expected frequency per batch
sample_weights = torch.tensor(1.0 / counts[y_train], dtype=torch.float64)
sampler = WeightedRandomSampler(sample_weights, num_samples=len(y_train), replacement=True)

num_workers = 2 if device.type == 'cuda' else 0
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=num_workers, pin_memory=(device.type=='cuda'))
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=num_workers, pin_memory=(device.type=='cuda'))
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=num_workers, pin_memory=(device.type=='cuda'))

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

In [ ]:
# ── Build EfficientNet-B3 ─────────────────────────────────────────────────
efn_weights = models.EfficientNet_B3_Weights.DEFAULT
model = models.efficientnet_b3(weights=efn_weights)

# Freeze entire backbone
for param in model.parameters():
    param.requires_grad = False

# Replace classifier head with a deeper one
in_features = model.classifier[-1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.4),
    nn.Linear(in_features, 512),
    nn.ReLU(),
    nn.Dropout(p=0.3),
    nn.Linear(512, num_classes),
)
model = model.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Phase 1 — trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)')

# Class-weighted loss (additional guard against imbalance)
cw = torch.tensor(1.0 / counts, dtype=torch.float32)
cw = (cw / cw.sum() * num_classes).to(device)
criterion = nn.CrossEntropyLoss(weight=cw, label_smoothing=LABEL_SMOOTH)

In [ ]:
# ── Helpers ───────────────────────────────────────────────────────────────
def evaluate(loader, desc=''):
    model.eval()
    preds_all, labels_all = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            p  = torch.argmax(model(xb), dim=1).cpu().numpy()
            preds_all.extend(p)
            labels_all.extend(yb.numpy())
    yt = np.array(labels_all)
    yp = np.array(preds_all)
    f1  = float(f1_score(yt, yp, average='macro', zero_division=0))
    acc = float(accuracy_score(yt, yp))
    return f1, acc, yt, yp


def run_phase(optimizer, scheduler, n_epochs, phase_name, patience):
    best_f1, best_state, best_ep, no_imp = -1.0, None, -1, 0
    log = []
    print(f'\n  {"Ep":>4} | {"Loss":>7} | {"ValF1":>7} | {"ValAcc":>7} | {"Time":>6}')
    print(f'  {"-"*43}')
    for ep in range(1, n_epochs + 1):
        model.train()
        t0 = time.time()
        total_loss, nb = 0.0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item(); nb += 1
        if scheduler: scheduler.step()
        avg_loss = total_loss / max(nb, 1)
        vf1, vacc, _, _ = evaluate(val_loader)
        elapsed = time.time() - t0
        print(f'  {ep:>4} | {avg_loss:>7.4f} | {vf1:>7.4f} | {vacc:>7.4f} | {elapsed:>5.1f}s')
        log.append({'epoch': ep, 'phase': phase_name, 'loss': round(avg_loss,4),
                    'val_f1': round(vf1,4), 'val_acc': round(vacc,4)})
        if vf1 > best_f1:
            best_f1    = vf1
            best_ep    = ep
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_imp     = 0
            print(f'    ✓ best val_f1={best_f1:.4f}')
        else:
            no_imp += 1
            if no_imp >= patience:
                print(f'  Early stop at epoch {ep}')
                break
    return best_f1, best_state, best_ep, log

In [ ]:
# ── Phase 1: Train head only (backbone frozen) ────────────────────────────
print('=' * 55)
print('PHASE 1 — Head only (backbone frozen)')
print('=' * 55)
opt1 = torch.optim.AdamW(model.classifier.parameters(), lr=LR_HEAD, weight_decay=1e-4)
sch1 = torch.optim.lr_scheduler.CosineAnnealingLR(opt1, T_max=PHASE1_EPOCHS)
f1_p1, state_p1, ep_p1, log_p1 = run_phase(opt1, sch1, PHASE1_EPOCHS, 'phase1', PHASE1_EPOCHS)
model.load_state_dict(state_p1)
print(f'\nPhase 1 best val F1: {f1_p1:.4f}')

In [ ]:
# ── Phase 2: Unfreeze last 3 feature blocks + fine-tune ───────────────────
print('=' * 55)
print('PHASE 2 — Fine-tune last 3 blocks + head')
print('=' * 55)

# EfficientNet-B3 has 9 feature blocks (0-8). Unfreeze 6, 7, 8.
for block_idx in [6, 7, 8]:
    for param in model.features[block_idx].parameters():
        param.requires_grad = True

trainable2 = sum(p.numel() for p in model.parameters() if p.requires_grad)
total2     = sum(p.numel() for p in model.parameters())
print(f'Phase 2 — trainable: {trainable2:,} / {total2:,} ({100*trainable2/total2:.1f}%)')

# Differential learning rates: lower LR for backbone, higher for head
opt2 = torch.optim.AdamW([
    {'params': model.features[6].parameters(), 'lr': LR_FINETUNE},
    {'params': model.features[7].parameters(), 'lr': LR_FINETUNE * 2},
    {'params': model.features[8].parameters(), 'lr': LR_FINETUNE * 3},
    {'params': model.classifier.parameters(),  'lr': LR_HEAD * 0.1},
], weight_decay=1e-4)
sch2 = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=PHASE2_EPOCHS)

f1_p2, state_p2, ep_p2, log_p2 = run_phase(opt2, sch2, PHASE2_EPOCHS, 'phase2', PATIENCE)
model.load_state_dict(state_p2)
print(f'\nPhase 2 best val F1: {f1_p2:.4f}')

In [ ]:
# ── Final evaluation ──────────────────────────────────────────────────────
print('=' * 55)
print('FINAL EVALUATION')
print('=' * 55)

# Evaluate on clean (no-augment) train set
eval_train_ds     = SkinDataset(X_train, y_train, transform=eval_tfm)
eval_train_loader = DataLoader(eval_train_ds, batch_size=BATCH_SIZE, shuffle=False,
                               num_workers=num_workers)

train_f1, train_acc, _, _   = evaluate(eval_train_loader)
val_f1,   val_acc,   _, _   = evaluate(val_loader)
test_f1,  test_acc,  yt, yp = evaluate(test_loader)
gap = train_f1 - val_f1

report = classification_report(
    yt, yp, labels=list(range(num_classes)),
    target_names=classes, output_dict=True, zero_division=0,
)
per_class = sorted(
    [(c, report[c]['f1-score']) for c in classes if c in report],
    key=lambda x: x[1]
)

print(f'\n  Train  macro-F1 = {train_f1:.4f}')
print(f'  Val    macro-F1 = {val_f1:.4f}')
print(f'  Test   acc={test_acc:.4f}  macro-F1={test_f1:.4f}')
print(f'  Train-Val Gap   = {gap:.4f}  (< 0.10 = well-fitted)')
print(f'  Baseline        = F1 0.449 | acc 0.5796')
print(f'  Improvement     = F1 {test_f1-0.449:+.4f} | acc {test_acc-0.5796:+.4f}')
print(f'\n  Per-class F1:')
for name, f1 in per_class:
    bar = '█' * int(f1 * 30)
    print(f'    {name:45s}: {f1:.4f}  {bar}')

In [ ]:
# ── Save artifacts ────────────────────────────────────────────────────────
OUT = Path('/kaggle/working')
all_log = log_p1 + log_p2

# 1. Model checkpoint — compatible with backend/guidance/services/image_model.py
checkpoint = {
    'backend': 'torchvision',
    'architecture': 'efficientnet_b3',
    'model_state_dict': {k: v.detach().cpu() for k, v in model.state_dict().items()},
    'num_classes': num_classes,
    'class_names': classes,
    'input_size': IMAGE_SIZE,
    'normalization': {'mean': MEAN, 'std': STD},
    'best_val_macro_f1': round(max(f1_p1, f1_p2), 4),
    'test_macro_f1': round(test_f1, 4),
    'test_accuracy': round(test_acc, 4),
    'train_macro_f1': round(train_f1, 4),
    'training_strategy': 'efficientnet_b3_2phase_finetune_224px_weighted_sampler',
    'dataset': 'dermamnist_64px_upsampled_224',
    'epoch_log': all_log,
}
torch.save(checkpoint, OUT / 'skin_cnn_torch.pt')

# 2. Label map
with open(OUT / 'image_labels.json', 'w') as f:
    json.dump(classes, f, indent=2)

# 3. Training metrics
metrics = {
    'dataset': 'dermamnist_64px_upsampled_224',
    'architecture': 'efficientnet_b3_2phase_finetune',
    'training_strategy': 'efficientnet_b3_2phase_finetune_224px_weighted_sampler',
    'train_samples': int(len(X_train)),
    'val_samples': int(len(X_val)),
    'test_samples': int(len(X_test)),
    'classes': num_classes,
    'image_size': IMAGE_SIZE,
    'train_macro_f1': round(train_f1, 4),
    'best_val_macro_f1': round(max(f1_p1, f1_p2), 4),
    'test_macro_f1': round(test_f1, 4),
    'test_accuracy': round(test_acc, 4),
    'train_val_gap': round(gap, 4),
    'baseline_test_f1': 0.449,
    'baseline_test_acc': 0.5796,
    'improvement_f1': round(test_f1 - 0.449, 4),
    'improvement_acc': round(test_acc - 0.5796, 4),
    'epoch_log': all_log,
    'classification_report': report,
}
with open(OUT / 'image_training_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print('Saved to /kaggle/working:')
print('  skin_cnn_torch.pt')
print('  image_labels.json')
print('  image_training_metrics.json')
print()
print('Next steps:')
print('  1. Download all 3 files from the Output tab')
print('  2. Copy to backend/models/ on your local machine')
print('  3. Restart Django: python manage.py runserver')